# Credit Risk Detection — Full MLOps Pipeline (University GPU Cluster)

**University MLOps Group 10**

This notebook is the **cluster-compatible** version of the Colab pipeline.  
It runs identically to `colab_full_pipeline.ipynb` but without any Google Colab dependencies (`google.colab`, `/content/` paths, `files.download`, Colab Secrets).

> **Before you start:**
> 1. Set `REPO_ROOT` in the Setup cell below to wherever the repo lives on the cluster (e.g. `~/MLOps-group-10` or `/scratch/username/MLOps-group-10`).
> 2. Set your `HF_TOKEN` and `WANDB_API_KEY` — either export them in your shell before launching Jupyter, or fill them in the Setup cell.

---
## Section 1: Setup

Check the GPU, set paths, install dependencies, and configure API keys.

In [ ]:
# Verify GPU availability
!nvidia-smi

In [ ]:
import os

# ── CONFIGURE THESE TWO LINES ────────────────────────────────────────────────
# Absolute path to the cloned repo on the cluster.
# Examples:
#   REPO_ROOT = os.path.expanduser("~/MLOps-group-10")
#   REPO_ROOT = "/scratch/s1234567/MLOps-group-10"
REPO_ROOT = os.path.expanduser("~/MLOps-group-10")

# API keys — if you already exported them in your shell (e.g. in ~/.bashrc)
# leave these as empty strings and the os.environ.get() fallback below handles it.
HF_TOKEN      = ""   # or: "hf_xxxx"
WANDB_API_KEY = ""   # or: "your_wandb_key"
# ─────────────────────────────────────────────────────────────────────────────

# Use shell exports if the variables above are left blank
HF_TOKEN      = HF_TOKEN      or os.environ.get("HF_TOKEN", "")
WANDB_API_KEY = WANDB_API_KEY or os.environ.get("WANDB_API_KEY", "")

if not HF_TOKEN:
    print("WARNING: HF_TOKEN is not set. Model download from HuggingFace will fail.")
if not WANDB_API_KEY:
    print("WARNING: WANDB_API_KEY is not set. W&B logging will be disabled.")

os.environ["HF_TOKEN"]               = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
os.environ["WANDB_API_KEY"]          = WANDB_API_KEY

print(f"REPO_ROOT : {REPO_ROOT}")
print(f"HF_TOKEN  : {'set' if HF_TOKEN else 'NOT SET'}")
print(f"W&B key   : {'set' if WANDB_API_KEY else 'NOT SET'}")

In [ ]:
# Clone or update the repo — edit the URL to your actual GitHub repo.
GITHUB_REPO_URL = "https://github.com/YOUR_ORG/MLOps-group-10"

if not os.path.isdir(REPO_ROOT):
    print(f"Cloning {GITHUB_REPO_URL} -> {REPO_ROOT} ...")
    !git clone {GITHUB_REPO_URL} {REPO_ROOT}
    print("Clone complete.")
else:
    print(f"Repo already exists at {REPO_ROOT}, pulling latest...")
    !git -C {REPO_ROOT} pull

In [ ]:
# Install dependencies.
# On most clusters pip installs into your user site-packages (~/.local).
# If the cluster uses modules/conda, activate your environment first.
print("Installing dependencies...")
!pip install -q -r {REPO_ROOT}/requirements.txt
!pip install -q bitsandbytes>=0.41.0   # ensure CUDA-enabled build
print("Done.")

In [ ]:
# Change into the repo root so all relative paths in scripts resolve correctly.
os.chdir(REPO_ROOT)
print(f"Working directory: {os.getcwd()}")
!ls -la

---
## Section 2: Weights & Biases Login

In [ ]:
import wandb

# WANDB_API_KEY is already in the environment — login is non-interactive.
wandb.login()
print(f"W&B version: {wandb.__version__}")
print("Logged in to Weights & Biases.")

---
## Section 3: Verify Pre-Processed Data

In [ ]:
import os
import pandas as pd

required_paths = {
    "Labels CSV"    : f"{REPO_ROOT}/data/labels/company_labels.csv",
    "FAISS index"   : f"{REPO_ROOT}/data/embeddings/faiss.index",
    "Chunk metadata": f"{REPO_ROOT}/data/embeddings/chunk_metadata.json",
    "Train JSONL"   : f"{REPO_ROOT}/data/finetune/train.jsonl",
    "Val JSONL"     : f"{REPO_ROOT}/data/finetune/val.jsonl",
    "Test JSONL"    : f"{REPO_ROOT}/data/finetune/test.jsonl",
}

all_ok = True
for name, path in required_paths.items():
    exists = os.path.isfile(path)
    status = "OK" if exists else "MISSING"
    print(f"  [{status}] {name}: {path}")
    if not exists:
        all_ok = False

processed_dir = f"{REPO_ROOT}/data/processed"
if os.path.isdir(processed_dir):
    processed_files = [f for f in os.listdir(processed_dir) if f.endswith(".json") and "sections" in f]
    print(f"\n  [OK] Processed sections: {len(processed_files)} companies")
else:
    print(f"\n  [MISSING] Processed directory: {processed_dir}")
    all_ok = False

print("\nAll data files present." if all_ok else "\nWARNING: Some files are missing — re-run the data pipeline scripts.")

In [ ]:
from IPython.display import display

labels_df = pd.read_csv(f"{REPO_ROOT}/data/labels/company_labels.csv")

def style_risk(val):
    colors = {"low": "#d4edda", "medium": "#fff3cd", "high": "#f8d7da"}
    return f"background-color: {colors.get(str(val).lower(), '')}"

print(f"Dataset: {len(labels_df)} companies")
display(
    labels_df.style
    .applymap(style_risk, subset=["risk_category"])
    .format({"risk_score": "{:.0f}"})
    .set_caption("Company Credit Risk Labels (Ground Truth)")
)

In [ ]:
import json
import faiss

index = faiss.read_index(f"{REPO_ROOT}/data/embeddings/faiss.index")
with open(f"{REPO_ROOT}/data/embeddings/chunk_metadata.json") as f:
    metadata = json.load(f)

print("FAISS Index Statistics")
print(f"  Total vectors  : {index.ntotal}")
print(f"  Dimension      : {index.d}")
print(f"  Metadata chunks: {len(metadata)}")

if metadata:
    sample = metadata[0]
    print(f"\nSample chunk: ticker={sample.get('ticker','N/A')} | section={sample.get('section','N/A')}")
    print(f"  text: {str(sample.get('text',''))[:120]}...")

---
## Section 4: Baseline Inference

Direct Mistral-7B prompting — no retrieval, no fine-tuning.

In [ ]:
!python3 scripts/run_inference_all.py --approach baseline

In [ ]:
baseline_path = f"{REPO_ROOT}/data/results/baseline_predictions.csv"

if os.path.isfile(baseline_path):
    baseline_df = pd.read_csv(baseline_path)
    display_cols = ["ticker", "company_name", "predicted_score", "predicted_label", "risk_level"]

    def style_risk_level(val):
        colors = {"low": "#d4edda", "medium": "#fff3cd", "high": "#f8d7da"}
        return f"background-color: {colors.get(str(val).lower(), '')}"

    print(f"Baseline predictions: {len(baseline_df)} companies")
    display(
        baseline_df[display_cols].style
        .applymap(style_risk_level, subset=["risk_level"])
        .format({"predicted_score": "{:.1f}"})
        .set_caption("Baseline Inference Results")
    )
else:
    print(f"Not found: {baseline_path}")

---
## Section 5: RAG Inference

Mistral-7B with FAISS retrieval — relevant 10-K passages are prepended to the prompt.

In [ ]:
!python3 scripts/run_inference_all.py --approach rag

In [ ]:
rag_path = f"{REPO_ROOT}/data/results/rag_predictions.csv"

if os.path.isfile(rag_path):
    rag_df = pd.read_csv(rag_path)
    display_cols = ["ticker", "company_name", "predicted_score", "predicted_label", "risk_level"]
    print(f"RAG predictions: {len(rag_df)} companies")
    display(
        rag_df[display_cols].style
        .applymap(style_risk_level, subset=["risk_level"])
        .format({"predicted_score": "{:.1f}"})
        .set_caption("RAG Inference Results")
    )

    # Baseline vs RAG score delta
    if os.path.isfile(baseline_path):
        compare = pd.read_csv(baseline_path)[["ticker", "predicted_score"]].rename(columns={"predicted_score": "baseline_score"})
        compare = compare.merge(rag_df[["ticker", "predicted_score"]].rename(columns={"predicted_score": "rag_score"}), on="ticker")
        compare["delta"] = compare["rag_score"] - compare["baseline_score"]
        print("\nBaseline vs RAG score delta:")
        display(compare.style.format("{:.1f}", subset=["baseline_score", "rag_score", "delta"]))
else:
    print(f"Not found: {rag_path}")

---
## Section 6: Generate Fine-Tune Data (if missing)

In [ ]:
train_path = f"{REPO_ROOT}/data/finetune/train.jsonl"
val_path   = f"{REPO_ROOT}/data/finetune/val.jsonl"
test_path  = f"{REPO_ROOT}/data/finetune/test.jsonl"

FORCE_REGENERATE = False

if not all(os.path.isfile(p) for p in [train_path, val_path, test_path]) or FORCE_REGENERATE:
    print("Generating fine-tune data...")
    !python3 scripts/generate_finetune_data.py
else:
    print("Fine-tune data already present — skipping.")

for split, path in [("train", train_path), ("val", val_path), ("test", test_path)]:
    if os.path.isfile(path):
        with open(path) as f:
            n = sum(1 for line in f if line.strip())
        print(f"  {split:5s}: {n} examples")
    else:
        print(f"  {split:5s}: MISSING")

---
## Section 7: LoRA Fine-Tuning

Trains three LoRA configurations for the Extra Mile comparison:

| Config | r | alpha | Target modules |
|--------|---|-------|----------------|
| r8     | 8 | 16 | q, v only |
| r16 (baseline) | 16 | 32 | q, k, v, o |
| r32    | 32 | 64 | q, k, v, o, up, down |

Each run logs to W&B under project `credit-risk-detection`.

In [ ]:
# Run LoRA r=16 (baseline config)
!python3 scripts/train_lora.py --config configs/lora_config.yaml

In [ ]:
# Run LoRA r=8 (smaller, faster)
!python3 scripts/train_lora.py --config configs/lora_config_r8.yaml

In [ ]:
# Run LoRA r=32 (larger, more expressive)
!python3 scripts/train_lora.py --config configs/lora_config_r32.yaml

In [ ]:
# Verify adapters were saved
for adapter_dir in [
    f"{REPO_ROOT}/data/models/lora_adapter_r8",
    f"{REPO_ROOT}/data/models/lora_adapter",
    f"{REPO_ROOT}/data/models/lora_adapter_r32",
]:
    final = os.path.join(adapter_dir, "final_adapter")
    target = final if os.path.isdir(final) else adapter_dir
    if os.path.isdir(target):
        files_list = os.listdir(target)
        print(f"  {target}")
        for fname in sorted(files_list):
            sz = os.path.getsize(os.path.join(target, fname)) / 1e6
            print(f"    {fname}  ({sz:.1f} MB)")
    else:
        print(f"  NOT FOUND: {target}")

In [ ]:
# Pull training loss curves from W&B
try:
    import wandb
    import matplotlib.pyplot as plt

    api = wandb.Api()
    fig, ax = plt.subplots(figsize=(9, 5))

    for run_name, color in [("lora-r8", "blue"), ("lora-sft", "green"), ("lora-r32", "red")]:
        runs = api.runs("credit-risk-detection", filters={"display_name": run_name})
        runs = sorted(runs, key=lambda r: r.created_at, reverse=True)
        if runs:
            history = runs[0].history(keys=["train/loss"], pandas=True)
            if not history.empty and "train/loss" in history.columns:
                ax.plot(history["train/loss"].dropna(), label=run_name, color=color)

    ax.set_xlabel("Step")
    ax.set_ylabel("Train Loss")
    ax.set_title("LoRA Training Loss — r8 vs r16 vs r32")
    ax.legend()
    plt.tight_layout()
    out_path = f"{REPO_ROOT}/data/results/lora_loss_curve.png"
    plt.savefig(out_path, dpi=150)
    plt.show()
    print(f"Saved to {out_path}")
except Exception as e:
    print(f"Could not fetch W&B loss curves: {e}")
    print("Check your W&B dashboard at https://wandb.ai")

---
## Section 8: LoRA Inference

Runs inference using the best LoRA adapter (r=16 by default — swap `ADAPTER_PATH` to compare configs).

In [ ]:
import os

# Adapter paths for all 3 LoRA configs
LORA_CONFIGS = {
    "lora_r16": f"{REPO_ROOT}/data/models/lora_adapter",
    "lora_r8" : f"{REPO_ROOT}/data/models/lora_adapter_r8",
    "lora_r32": f"{REPO_ROOT}/data/models/lora_adapter_r32",
}

for approach, base_dir in LORA_CONFIGS.items():
    # Prefer final_adapter subdirectory if it exists
    final = os.path.join(base_dir, "final_adapter")
    adapter_path = final if os.path.isdir(final) else base_dir

    if not os.path.isdir(adapter_path):
        print(f"[SKIP] {approach}: adapter not found at {adapter_path}")
        continue

    print(f"\nRunning inference for {approach} — adapter: {adapter_path}")
    # Output: data/results/{approach}_predictions.csv  (e.g. lora_r16_predictions.csv)
    !python3 scripts/run_inference_all.py --approach {approach} --adapter-path {adapter_path}


In [ ]:
# Display LoRA predictions for all 3 configs
import pandas as pd
from IPython.display import display
import os

display_cols = ["ticker", "company_name", "predicted_score", "predicted_label", "risk_level"]

lora_approaches = {
    "lora_r8" : "LoRA r=8  (q,v only)",
    "lora_r16": "LoRA r=16 (q,k,v,o)",
    "lora_r32": "LoRA r=32 (q,k,v,o,up,down)",
}

for approach, label in lora_approaches.items():
    path = f"{REPO_ROOT}/data/results/{approach}_predictions.csv"
    if os.path.isfile(path):
        df = pd.read_csv(path)
        available_cols = [c for c in display_cols if c in df.columns]
        print(f"\n{label}: {len(df)} companies")
        display(
            df[available_cols].style
            .map(style_risk_level, subset=["risk_level"] if "risk_level" in available_cols else [])
            .format({"predicted_score": "{:.1f}"})
            .set_caption(f"LoRA Inference Results — {label}")
        )
    else:
        print(f"[MISSING] {label}: {path}")


---
## Section 9: Full Evaluation

Compares Baseline, RAG, LoRA, and Altman Z-Score. Outputs:
- `data/results/metrics_summary.csv`
- `data/results/roc_curves.png`

In [ ]:
!python3 scripts/evaluate_all.py

In [ ]:
metrics_path = f"{REPO_ROOT}/data/results/metrics_summary.csv"

if os.path.isfile(metrics_path):
    metrics_df = pd.read_csv(metrics_path)
    numeric_cols = metrics_df.select_dtypes(include="number").columns.tolist()

    def highlight_best(s):
        if s.name not in numeric_cols:
            return [""] * len(s)
        is_best = s == s.max()
        return ["background-color: #d4edda; font-weight: bold" if v else "" for v in is_best]

    display(
        metrics_df.style
        .apply(highlight_best)
        .format({col: "{:.4f}" for col in numeric_cols})
        .set_caption("Comparison of Credit Risk Detection Approaches")
        .set_table_styles([{"selector": "th", "props": [("background-color", "#343a40"), ("color", "white")]}])
    )
else:
    print(f"Not found: {metrics_path}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

roc_path = f"{REPO_ROOT}/data/results/roc_curves.png"

if os.path.isfile(roc_path):
    img = mpimg.imread(roc_path)
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.imshow(img)
    ax.axis("off")
    ax.set_title("ROC Curves — All Approaches", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()
else:
    print(f"Not found: {roc_path}")

In [ ]:
# Show a per-company prediction comparison across all approaches
import pandas as pd
from IPython.display import display
import os

RESULTS_DIR = f"{REPO_ROOT}/data/results"

labels = pd.read_csv(f"{REPO_ROOT}/data/labels/company_labels.csv")[["ticker", "label", "risk_category"]].rename(
    columns={"label": "true_label", "risk_category": "true_risk"}
)
comparison = labels.copy()

for approach in ["baseline", "rag", "lora_r8", "lora_r16", "lora_r32"]:
    pred_path = os.path.join(RESULTS_DIR, f"{approach}_predictions.csv")
    if os.path.isfile(pred_path):
        df = pd.read_csv(pred_path)[["ticker", "predicted_label", "predicted_score"]]
        df = df.rename(columns={
            "predicted_label": f"{approach}_pred",
            "predicted_score" : f"{approach}_score",
        })
        comparison = comparison.merge(df, on="ticker", how="left")

for approach in ["baseline", "rag", "lora_r8", "lora_r16", "lora_r32"]:
    col = f"{approach}_pred"
    if col in comparison.columns:
        comparison[f"{approach}_correct"] = (comparison[col] == comparison["true_label"])

def highlight_correct(val):
    if val is True:
        return "background-color: #d4edda"
    elif val is False:
        return "background-color: #f8d7da"
    return ""

correct_cols = [c for c in comparison.columns if c.endswith("_correct")]
print("Per-Company Prediction Comparison (green = correct, red = wrong)")
display(comparison.style.map(highlight_correct, subset=correct_cols))


---
## Section 10: Results Location

On the cluster, results are saved to disk — use `scp` or the cluster's file transfer portal to retrieve them.

In [ ]:
import os

print("=" * 60)
print("  PIPELINE COMPLETE")
print("=" * 60)

output_files = {
    "Baseline predictions" : f"{REPO_ROOT}/data/results/baseline_predictions.csv",
    "RAG predictions"      : f"{REPO_ROOT}/data/results/rag_predictions.csv",
    "LoRA r=8  predictions": f"{REPO_ROOT}/data/results/lora_r8_predictions.csv",
    "LoRA r=16 predictions": f"{REPO_ROOT}/data/results/lora_r16_predictions.csv",
    "LoRA r=32 predictions": f"{REPO_ROOT}/data/results/lora_r32_predictions.csv",
    "Altman Z-Score"       : f"{REPO_ROOT}/data/results/altman_zscore_predictions.csv",
    "Metrics summary"      : f"{REPO_ROOT}/data/results/metrics_summary.csv",
    "ROC curves"           : f"{REPO_ROOT}/data/results/roc_curves.png",
    "LoRA adapter r=16"    : f"{REPO_ROOT}/data/models/lora_adapter",
    "LoRA adapter r=8"     : f"{REPO_ROOT}/data/models/lora_adapter_r8",
    "LoRA adapter r=32"    : f"{REPO_ROOT}/data/models/lora_adapter_r32",
}

print("\nOutput files:")
for name, path in output_files.items():
    exists = os.path.isfile(path) or os.path.isdir(path)
    status = "OK" if exists else "MISSING"
    print(f"  [{status:7s}] {name}")

print(f"\nTo copy results to your local machine, run from your laptop:")
print(f"  scp -r <username>@<cluster-hostname>:{REPO_ROOT}/data/results/ ./results/")
print(f"  scp -r <username>@<cluster-hostname>:{REPO_ROOT}/data/models/ ./models/")
print(f"\nW&B Dashboard: https://wandb.ai → project: credit-risk-detection")
print("=" * 60)
